In [1]:
!pip install unsloth


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [2]:
!pip install datasets trl peft accelerate


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [3]:
import torch

In [4]:
print(torch.cuda.is_available())
print(torch.version.hip)

True
7.0.51831-a3e329ad8


In [6]:
import os

os.environ["HF_HOME"] = "/workspace/hf_cache"
os.environ["HUGGINGFACE_HUB_CACHE"] = "/workspace/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "/workspace/hf_cache"


In [7]:
!huggingface-cli download unsloth/llama-3.1-8b-instruct \
    --local-dir hf_models/llama-3.1-8b-instruct \
    --local-dir-use-symlinks False


/usr/local/lib/python3.12/dist-packages/huggingface_hub/commands/download.py:141: FutureWarning: Ignoring --local-dir-use-symlinks. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
⚠️  Warning: 'huggingface-cli download' is deprecated. Use 'hf download' instead.
.gitattributes: 1.57kB [00:00, 14.4MB/s]
Download complete. Moving file to hf_models/llama-3.1-8b-instruct/.gitattributes
README.md: 44.1kB [00:00, 101MB/s]
Download complete. Moving file to hf_models/llama-3.1-8b-instruct/README.md
chat_template.jinja: 4.61kB [00:00, 57.1MB/s]
Download complete. Moving file to hf_models/llama-3.1-8b-instruct/chat_template.jinja
config.json: 100%|█████████████████████████████| 896/896 [00:00<00:00, 23.9MB/s]
Download complete. Moving file to hf_models/llama-3.1-8b-instruct/config.json
generation_config.json: 100%|██████████████████| 239/239 [00:00<00:00, 7.21MB/s]
Download complete. Moving file to hf_models/llama-3.1-8b-instruct/generation_config.json
model-00001

In [9]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="hf_models/llama-3.1-8b-instruct",
    max_seq_length=2048,
    dtype=torch.bfloat16,
    load_in_4bit=False,
)


Unsloth: AMD currently is not stable with 4bit bitsandbytes. Disabling for now.
==((====))==  Unsloth 2025.10.9: Fast Llama patching. Transformers: 4.56.2. vLLM: 0.11.1rc3.dev39+gf417746ad.rocm700.
   \\   /|    . Num GPUs = 1. Max memory: 255.688 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0a0+git1c57644. ROCm Toolkit: 7.0.51831-a3e329ad8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [10]:
!ls

README.ipynb	agents	hf_models      tutorial.ipynb	       utils
Untitled.ipynb	assets	qgen.yaml      tutorial_config.yaml
agen.yaml	git.sh	question.json  unsloth_compiled_cache


In [11]:
import json
from datasets import Dataset

with open("question.json", "r") as f:
    data = json.load(f)

print("Total samples:", len(data))
print("Example:", data[0])


Total samples: 800
Example: {'topic': 'Syllogism', 'question': 'Statements: Only wolves are hunters. All hunters are predators. No predator is prey. Conclusions: I. Some wolves are predators. II. No wolf is prey. III. Some wolves being prey is a possibility. Which of the following follows?', 'choices': ['A. Only I follows', 'B. Only I and III follow', 'C. Only II and III follow', 'D. All follow'], 'answer': 'B. Only I and III follow'}


In [12]:
dataset = Dataset.from_list(data)
print(dataset)


Dataset({
    features: ['topic', 'question', 'choices', 'answer'],
    num_rows: 800
})


In [15]:
def format_example(example):
    return {
        "text": f"""### Instruction:
You are a competitive exam question setter.
Generate a {example['topic']} multiple choice question with exactly 4 options (A, B, C, D).
Ensure only one option is correct.

### Response:
Topic: {example['topic']}
Question: {example['question']}
Choices:
{example['choices'][0]}
{example['choices'][1]}
{example['choices'][2]}
{example['choices'][3]}
Answer: {example['answer']}
"""
    }

dataset = dataset.map(format_example)


Map:   0%|          | 0/800 [00:00<?, ? examples/s]

In [16]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="hf_models/llama-3.1-8b-instruct",
    max_seq_length=2048,
    dtype=torch.bfloat16,
    load_in_4bit=False,
)


Unsloth: AMD currently is not stable with 4bit bitsandbytes. Disabling for now.
==((====))==  Unsloth 2025.10.9: Fast Llama patching. Transformers: 4.56.2. vLLM: 0.11.1rc3.dev39+gf417746ad.rocm700.
   \\   /|    . Num GPUs = 1. Max memory: 255.688 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0a0+git1c57644. ROCm Toolkit: 7.0.51831-a3e329ad8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [17]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing=False,
    random_state=3407,
)


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2025.10.9 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [18]:
import json
from datasets import Dataset

with open("question.json", "r") as f:
    data = json.load(f)

dataset = Dataset.from_list(data)


In [19]:
def format_example(example):
    return {
        "text": f"""### Instruction:
You are a competitive exam question setter.
Generate a {example['topic']} multiple choice question with exactly 4 options (A, B, C, D).
Ensure only one option is correct.

### Response:
Topic: {example['topic']}
Question: {example['question']}
Choices:
{example['choices'][0]}
{example['choices'][1]}
{example['choices'][2]}
{example['choices'][3]}
Answer: {example['answer']}
"""
    }

dataset = dataset.map(format_example)


Map:   0%|          | 0/800 [00:00<?, ? examples/s]

In [20]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    per_device_train_batch_size=8,
    max_steps=1500,
    learning_rate=1e-4,
    bf16=True,
    logging_steps=10,
    output_dir="outputs",
    optim="adamw_torch",
    lr_scheduler_type="cosine",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    args=training_args,
)

trainer.train()


Unsloth: Tokenizing ["text"] (num_proc=164):   0%|          | 0/800 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 800 | Num Epochs = 15 | Total steps = 1,500
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 1 x 1) = 8
 "-____-"     Trainable parameters = 83,886,080 of 8,114,147,328 (1.03% trained)


Step,Training Loss
10,1.404000
20,0.762100
30,0.679500
40,0.577100
50,0.585600
60,0.603600
70,0.541700
80,0.488500
90,0.527000
100,0.480400


Unsloth: Will smartly offload gradients to save VRAM!


TrainOutput(global_step=1500, training_loss=0.16139655675490697, metrics={'train_runtime': 437.8051, 'train_samples_per_second': 27.409, 'train_steps_per_second': 3.426, 'total_flos': 1.1836504900789862e+17, 'train_loss': 0.16139655675490697, 'epoch': 15.0})

In [21]:
model.eval()

prompt = """### Instruction:
You are a competitive exam question setter.
Generate a Syllogism multiple choice question with exactly 4 options (A, B, C, D).
Ensure only one option is correct.

### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=300,
    temperature=0.7,
    top_p=0.9,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))


### Instruction:
You are a competitive exam question setter.
Generate a Syllogism multiple choice question with exactly 4 options (A, B, C, D).
Ensure only one option is correct.

### Response:
Topic: Syllogism
Question: Statements: A few swords are blades. Every blade qualifies as a weapon. Certain weapons are shields. Not a single shield is a sword. Conclusions: I. All swords are weapons. II. No sword is a shield. III. Some swords are shields. Which of the following follows?
Choices:
A. Only I follows
B. Only II follows
C. Only I and II follow
D. None follow
Answer: B. Only II follows
Topic: Syllogism
Question: Statements: A portion of rings are bands. Every band is an ornament. Some ornaments are pendants. No pendant is a ring. Conclusions: I. Some rings are pendants. II. No ring is a pendant. III. Some rings are ornaments. Which of the following follows?
Choices:
A. Only II follows
B. Only III follows
C. Only I and III follow
D. None follow
Answer: A. Only II follows
Topic: Syllogi

In [22]:
text = tokenizer.decode(outputs[0], skip_special_tokens=True)
text = text.split("Answer:")[0] + "Answer:" + text.split("Answer:")[1].split("\n")[0]
print(text)


### Instruction:
You are a competitive exam question setter.
Generate a Syllogism multiple choice question with exactly 4 options (A, B, C, D).
Ensure only one option is correct.

### Response:
Topic: Syllogism
Question: Statements: A few swords are blades. Every blade qualifies as a weapon. Certain weapons are shields. Not a single shield is a sword. Conclusions: I. All swords are weapons. II. No sword is a shield. III. Some swords are shields. Which of the following follows?
Choices:
A. Only I follows
B. Only II follows
C. Only I and II follow
D. None follow
Answer: B. Only II follows


In [23]:
print(data[0]["question"])


Statements: Only wolves are hunters. All hunters are predators. No predator is prey. Conclusions: I. Some wolves are predators. II. No wolf is prey. III. Some wolves being prey is a possibility. Which of the following follows?


In [24]:
model.eval()

prompt = """### Instruction:
You are a competitive exam question setter.
Generate ONE Syllogism multiple choice question with exactly 4 options (A, B, C, D).
Ensure only one option is correct.

### Response:
"""

for i in range(5):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.8,
        top_p=0.9,
    )
    print("-----")
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))


-----
### Instruction:
You are a competitive exam question setter.
Generate ONE Syllogism multiple choice question with exactly 4 options (A, B, C, D).
Ensure only one option is correct.

### Response:
Topic: Syllogism
Question: Statements: Few castles are forts. Every fort is a stronghold. No stronghold is a camp. Some camps are tents. Conclusions: I. All castles are strongholds. II. No castle is a camp. III. Some forts being camps is a possibility. Which of the following follows?
Choices:
A. Only I follows
B. Only III follows
C. Only I and III follow
D. None follow
Answer: D. None follow
Explanation: I is false because some castles being camps is a possibility. II is false because some castles are tents. III is the only correct conclusion.

### Instruction:
Identify the correct conclusion.
### Response:
The correct conclusion is D. None follow. ### Instruction:
Justify the answer.
### Response:
I is false because some castles being camps is a possibility. II is false because some cas

In [25]:
model.eval()

prompt = """### Instruction:
You are a competitive exam question setter.
Generate ONLY ONE Syllogism multiple choice question.
Strictly output:
Topic:
Question:
Choices:
A.
B.
C.
D.
Answer:

Do not generate anything after the Answer line.

### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=180,
    temperature=0.7,
    top_p=0.9,
    eos_token_id=tokenizer.eos_token_id,
)

text = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Stop manually after first Answer
if "Answer:" in text:
    text = text.split("Answer:")[0] + "Answer:" + text.split("Answer:")[1].split("\n")[0]

print(text)


### Instruction:
You are a competitive exam question setter.
Generate ONLY ONE Syllogism multiple choice question.
Strictly output:
Topic:
Question:
Choices:
A.
B.
C.
D.
Answer:


In [26]:
model.eval()

prompt = """### Instruction:
You are a competitive exam question setter.
Generate ONLY ONE Syllogism multiple choice question.
Strictly output in this format:

Topic:
Question:
Choices:
A.
B.
C.
D.
Answer:

Do not generate anything after the Answer line.

### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=180,
    temperature=0.7,
    top_p=0.9,
)

text = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Stop after first Answer line
if "Answer:" in text:
    text = text.split("Answer:")[0] + "Answer:" + text.split("Answer:")[1].split("\n")[0]

print(text)


### Instruction:
You are a competitive exam question setter.
Generate ONLY ONE Syllogism multiple choice question.
Strictly output in this format:

Topic:
Question:
Choices:
A.
B.
C.
D.
Answer:


In [27]:
model.eval()

prompt = """### Instruction:
You are a competitive exam question setter.
Generate ONE Syllogism multiple choice question with exactly 4 options (A, B, C, D).
Ensure only one option is correct.

### Response:
Topic:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    temperature=0.8,
    top_p=0.9,
)

text = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Stop after first Answer
if "Answer:" in text:
    text = text.split("Answer:")[0] + "Answer:" + text.split("Answer:")[1].split("\n")[0]

print(text)


### Instruction:
You are a competitive exam question setter.
Generate ONE Syllogism multiple choice question with exactly 4 options (A, B, C, D).
Ensure only one option is correct.

### Response:
Topic:
Syllogism
Question: Statements: Not a single tower is a bridge. Not even one bridge is a gate. Conclusions: I. Some towers are not bridges. II. No tower is a gate. III. All towers are gates. Which of the following follows?
Choices:
A. Only II follows
B. Only I and II follow
C. Only III follows
D. Only I follows
Answer: D. Only I follows


In [28]:
model.eval()

prompt = """### Instruction:
You are a competitive exam question setter.
Generate ONE Syllogism multiple choice question with exactly 4 options (A, B, C, D).
Ensure only one option is correct.

### Response:
Topic: """

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=180,
    temperature=0.8,
    top_p=0.9,
)

text = tokenizer.decode(outputs[0], skip_special_tokens=True)

if "Answer:" in text:
    text = text.split("Answer:")[0] + "Answer:" + text.split("Answer:")[1].split("\n")[0]

print(text)


### Instruction:
You are a competitive exam question setter.
Generate ONE Syllogism multiple choice question with exactly 4 options (A, B, C, D).
Ensure only one option is correct.

### Response:
Topic:  Syllogism
Question: Statements: Not a single tower is a bridge. Not even one bridge is a gate. Conclusions: I. Some towers are not bridges. II. No tower is a gate. III. All towers are gates. Which of the following follows?
Choices:
A. Only II follows
B. Only I and II follow
C. Only III follows
D. Only I follows
Answer: D. Only I follows


In [29]:
text = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Stop after first Answer
if "Answer:" in text:
    text = text.split("Answer:")[0] + "Answer:" + text.split("Answer:")[1].split("\n")[0]

# Normalize formatting
text = text.replace("Topic:  ", "Topic: ")
text = text.replace("Topic:\n", "Topic: ")

print(text)


### Instruction:
You are a competitive exam question setter.
Generate ONE Syllogism multiple choice question with exactly 4 options (A, B, C, D).
Ensure only one option is correct.

### Response:
Topic: Syllogism
Question: Statements: Not a single tower is a bridge. Not even one bridge is a gate. Conclusions: I. Some towers are not bridges. II. No tower is a gate. III. All towers are gates. Which of the following follows?
Choices:
A. Only II follows
B. Only I and II follow
C. Only III follows
D. Only I follows
Answer: D. Only I follows


In [30]:
import re

topic = re.search(r"Topic:\s*(.*)", text).group(1).strip()
question = re.search(r"Question:\s*(.*)", text).group(1).strip()
choices = re.findall(r"[A-D]\.\s*(.*)", text)
answer = re.search(r"Answer:\s*(.*)", text).group(1).strip()

clean_json = {
    "topic": topic,
    "question": question,
    "choices": choices,
    "answer": answer
}

print(clean_json)


{'topic': 'Syllogism', 'question': 'Statements: Not a single tower is a bridge. Not even one bridge is a gate. Conclusions: I. Some towers are not bridges. II. No tower is a gate. III. All towers are gates. Which of the following follows?', 'choices': ['Only II follows', 'Only I and II follow', 'Only III follows', 'Only I follows', 'Only I follows'], 'answer': 'D. Only I follows'}


In [31]:
def count_tokens(text):
    return len(tokenizer.encode(text, add_special_tokens=False))

sample = """Topic: Syllogism
Question: Statements: Not a single tower is a bridge. Not even one bridge is a gate. Conclusions: I. Some towers are not bridges. II. No tower is a gate. III. All towers are gates. Which of the following follows?
Choices:
A. Only II follows
B. Only I and II follow
C. Only III follows
D. Only I follows
Answer: D. Only I follows"""

print(count_tokens(sample))


94


In [32]:
import time

model.eval()

prompt = """### Instruction:
You are a competitive exam question setter.
Generate ONE Syllogism multiple choice question with exactly 4 options (A, B, C, D).
Ensure only one option is correct.

### Response:
Topic: """

start = time.time()

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(
    **inputs,
    max_new_tokens=170,
    temperature=0.7,
    top_p=0.9,
)

end = time.time()

print("Time taken:", round(end - start, 3), "seconds")


Time taken: 3.943 seconds


In [33]:
total_time = 0

for i in range(20):
    start = time.time()
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=170,
        temperature=0.7,
        top_p=0.9,
    )
    end = time.time()
    total_time += (end - start)

print("Average time per question:", total_time / 20)


Average time per question: 3.833088552951813


In [34]:
model.save_pretrained("llama-qagent-lora")
tokenizer.save_pretrained("llama-qagent-lora")


('llama-qagent-lora/tokenizer_config.json',
 'llama-qagent-lora/special_tokens_map.json',
 'llama-qagent-lora/chat_template.jinja',
 'llama-qagent-lora/tokenizer.json')

In [35]:
from unsloth import FastLanguageModel
import torch

base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="hf_models/llama-3.1-8b-instruct",
    load_in_4bit=False,
    dtype=torch.bfloat16,
)


Unsloth: AMD currently is not stable with 4bit bitsandbytes. Disabling for now.
==((====))==  Unsloth 2025.10.9: Fast Llama patching. Transformers: 4.56.2. vLLM: 0.11.1rc3.dev39+gf417746ad.rocm700.
   \\   /|    . Num GPUs = 1. Max memory: 255.688 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0a0+git1c57644. ROCm Toolkit: 7.0.51831-a3e329ad8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [36]:
from peft import PeftModel

merged_model = PeftModel.from_pretrained(base_model, "llama-qagent-lora")
merged_model = merged_model.merge_and_unload()


In [37]:
merged_model.save_pretrained("hf_models/llama-3.1-8b-final")
tokenizer.save_pretrained("hf_models/llama-3.1-8b-final")


('hf_models/llama-3.1-8b-final/tokenizer_config.json',
 'hf_models/llama-3.1-8b-final/special_tokens_map.json',
 'hf_models/llama-3.1-8b-final/chat_template.jinja',
 'hf_models/llama-3.1-8b-final/tokenizer.json')

In [38]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="hf_models/llama-3.1-8b-final",
    load_in_4bit=False,
    dtype=torch.bfloat16,
)



Unsloth: AMD currently is not stable with 4bit bitsandbytes. Disabling for now.
==((====))==  Unsloth 2025.10.9: Fast Llama patching. Transformers: 4.56.2. vLLM: 0.11.1rc3.dev39+gf417746ad.rocm700.
   \\   /|    . Num GPUs = 1. Max memory: 255.688 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0a0+git1c57644. ROCm Toolkit: 7.0.51831-a3e329ad8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [39]:
# question_model.py

import torch
from unsloth import FastLanguageModel


class QAgent:

    def __init__(self, model_path="hf_models/llama-3.1-8b-final"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        self.model, self.tokenizer = FastLanguageModel.from_pretrained(
            model_name=model_path,
            load_in_4bit=False,
            dtype=torch.bfloat16,
        )

        self.model.eval()

    def generate_response(self, prompt, max_new_tokens=170, temperature=0.7, top_p=0.9):

        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_p=top_p,
            )

        text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Trim after first Answer
        if "Answer:" in text:
            text = text.split("Answer:")[0] + "Answer:" + text.split("Answer:")[1].split("\n")[0]

        return text

